# VAM(1,2,2) hand-built **8-state** SystemModel — dam-break over a bump

The 7-state model in `vam_chorin_bump_handbuilt.py` carries gravity
through the hydrostatic-pressure tensor `P = g·h·(b + h)` with a
corrective NCP entry, and treats bathymetry `b` as a function-aux
variable.  It propagates the cosine bump cleanly on a flat bottom
and preserves lake-at-rest on a Gaussian bump (Audusse HR), but the
`q_U1` mode goes unstable when a dam-break shock excites it — even
at first order in time.

This notebook builds the alternative form that the OLD JAX prototype
(`web/tutorials/vam/simple.ipynb`) used, which ran the full
dam-break-over-bump experimental test cleanly:

  1. **Bathymetry `b` is a STATE** with trivial evolution `∂_t b = 0`.
  2. **Gravity is entirely in the NCP**:
     `B[xmom_j0, h, x] = B[xmom_j0, b, x] = g·h`.
     No contribution in `P`.  The momentum equation gets
     `g·h·(∂_x h + ∂_x b) = g·h·∂_x η`, the well-balanced form.
  3. **The pressure-flux** `h·P_k` lives in `F` (zeroed in `SM_pred`
     by the splitter's `xreplace({P_k: 0})`); the bottom-slope
     pressure-source terms live in `S`.

The Chorin pressure-projection structure stays the same — the
`split_simple` machinery copies the SystemModel, drops the
constraint rows for `SM_pred`, zeros the pressure-state symbols in
the remaining operators, and uses
`build_pressure_elliptic_block` for the dt-baked elliptic block
and the closed-form corrector update.

**Solver wiring.**  Audusse-HR `PositiveNonconservativeRusanov` via
`ChorinSplitVAMSolver` (predictor → pressure → corrector).  The
`_build_numerics` override excludes `b` from `scaled_q_indices`
because `b` is static bathymetry, not a momentum density that
needs the HR mass-preserving rescaling.

**Tests in this notebook.**

  * **Lake-at-rest** on a Gaussian bump — bit-exact preservation
    (machine-precision drift over 15 steps).
  * **Dam-break over a bump** at the literal Escalante 2024 setup
    (`L = 3`, 60 cells, `T_end = 20`) — runs to completion, free
    surface matches the experimental data within ~4 cm at the dip.

In [ ]:
import sys
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import zoomy_core.model.aux_boundary_conditions as AuxBC
from zoomy_core.misc.misc import Zstruct
from zoomy_core.mesh import BaseMesh
from zoomy_core.model.boundary_conditions import (
    BoundaryConditions, Extrapolation, Lambda,
)
from zoomy_core.model.initial_conditions import UserFunction
from zoomy_core.model.models.system_model import SystemModel
from zoomy_core.model.splitter import split_simple
from zoomy_core.fvm.solver_chorin_vam_numpy import ChorinSplitVAMSolver

## Digitized experimental data

Same digitization as `vam_chorin_bump_handbuilt.py` — embedded here
so the notebook is self-contained.

In [ ]:
ETA_EXP_X = np.array([
    -0.5928667563930013, -0.5430686406460297, -0.4946164199192463,
    -0.4448183041722746, -0.3990578734858681, -0.34522207267833105,
    -0.2981157469717362, -0.2510094212651413, -0.19851951547779273,
    -0.15141318977119783, -0.10430686406460293, -0.0531628532974428,
    -0.0006729475100942239, 0.04643337819650073, 0.09757738896366086,
    0.14737550471063254, 0.19851951547779279, 0.24562584118438757,
    0.29811574697173626, 0.34791386271870794, 0.3963660834454913,
    0.446164199192463, 0.49865410497981155, 0.5511440107671601,
])
ETA_EXP_Y = np.array([
    0.3418918918918919, 0.34121621621621623, 0.3398648648648649,
    0.3418918918918919, 0.3398648648648649, 0.3398648648648649,
    0.33851351351351355, 0.33783783783783783, 0.3337837837837838,
    0.32770270270270274, 0.322972972972973, 0.31486486486486487,
    0.3054054054054054, 0.29054054054054057, 0.26891891891891895,
    0.2425675675675676, 0.21621621621621623, 0.18581081081081083,
    0.15540540540540543, 0.13108108108108107, 0.10608108108108108,
    0.0918918918918919, 0.07297297297297298, 0.06554054054054054,
])

## Symbols and parameters

State vector (8 entries):

  `Q = [h, q_U0, q_U1, q_W0, q_W1, b, P_0, P_1]`

Equations (8 rows):

  `mass, xmom_j0, xmom_j1, zmom_j0, zmom_j1, b_eq, cont_j1, cont_j2`

In [ ]:
t = sp.Symbol("t", real=True)
x = sp.Symbol("x", real=True)

h    = sp.Symbol("h",    positive=True, real=True)
q_U0 = sp.Symbol("q_U0", real=True)
q_U1 = sp.Symbol("q_U1", real=True)
q_W0 = sp.Symbol("q_W0", real=True)
q_W1 = sp.Symbol("q_W1", real=True)
b    = sp.Symbol("b",    real=True)
P_0  = sp.Symbol("P_0",  real=True)
P_1  = sp.Symbol("P_1",  real=True)
state = [h, q_U0, q_U1, q_W0, q_W1, b, P_0, P_1]
n_state = 8
n_eq = 8

g   = sp.Symbol("g",   positive=True)
rho = sp.Symbol("rho", positive=True)
ez  = sp.Symbol("ez",  positive=True)
parameters = Zstruct(g=g, ez=ez, rho=rho)
parameters._symbolic_name = "p"
parameter_values = Zstruct(g=9.81, ez=1.0, rho=1000.0)

b_x    = sp.Symbol("b_x",    real=True)
h_x    = sp.Symbol("h_x",    real=True)
q_U0_x = sp.Symbol("q_U0_x", real=True)
q_U1_x = sp.Symbol("q_U1_x", real=True)
aux_state = [b_x, h_x, q_U0_x, q_U1_x]

# Primitive derivatives in conservative form (modal-conservative CoV).
U_0_x_expr = q_U0_x / h - q_U0 * h_x / h**2
U_1_x_expr = 3 * q_U1_x / h - 3 * q_U1 * h_x / h**2

## Operators

Residual convention:

  `M · ∂_t Q + ∂_x F + ∂_x P + B · ∂_x Q − S = 0`

In particular:

  * `F[xmom_j0] = h·P_0/ρ + q_U0²/h + 3·q_U1²/h` — convective +
    pressure-flux.  The pressure part vanishes in `SM_pred`.
  * `P = 0` everywhere — gravity moved into the NCP.
  * `B[xmom_j0, h, x] = B[xmom_j0, b, x] = g·h` — gravity acts on
    `∂_x η = ∂_x h + ∂_x b`.  When `h + b = const` the two NCP
    contributions cancel, giving bit-exact lake-at-rest at the
    symbolic level.
  * `B[xmom_j1, q_U1, x] = -q_U0/h` — chain's convective cross-term.
  * `B[xmom_j1, h, x] = (-P_0 + P_1/3)/ρ` — pressure cross-term;
    this is the j=1 row's analogue of the pressure-flux gradient.
  * `S` carries the bottom-slope pressure-source terms (zeroed in
    `SM_pred`) and the `q_U·U_x` / `b_x` cross-couplings the chain
    emits on the `zmom_j1` row.

In [ ]:
M = sp.eye(n_state)
M[6, 6] = 0
M[7, 7] = 0

F = sp.zeros(n_eq, 1)
F[0, 0] = q_U0
F[1, 0] = P_0 * h / rho + q_U0**2 / h + 3 * q_U1**2 / h
F[2, 0] = P_1 * h / (3 * rho) + 2 * q_U0 * q_U1 / h
F[3, 0] = (q_U0 * q_W0 + 3 * q_U1 * q_W1) / h
F[4, 0] = (
    2 * b_x * q_U0 * q_U1 + 6 * b_x * q_U1**2
    + 5 * q_U0 * q_W1 + 3 * q_U1 * q_W0 - 6 * q_U1 * q_W1
) / (5 * h)

P = sp.zeros(n_eq, 1)

B = sp.MutableDenseNDimArray.zeros(n_eq, n_state, 1)
B[1, 0, 0] = g * h
B[1, 5, 0] = g * h
B[2, 0, 0] = (-P_0 + P_1 / 3) / rho
B[2, 2, 0] = -q_U0 / h
B[6, 0, 0] = (-q_U0 + q_U1) / h
B[6, 1, 0] = sp.Integer(1)
B[6, 2, 0] = sp.Integer(1)
B[7, 0, 0] = (q_U0 - 3 * q_U1) / h
B[7, 1, 0] = sp.Integer(-1)

S = sp.zeros(n_eq, 1)
S[1, 0] = -2 * P_1 * b_x / rho
S[2, 0] = 2 * b_x * (P_0 - P_1) / rho
S[3, 0] = 2 * P_1 / rho
S[4, 0] = (
    -2 * P_0 / rho + 2 * P_1 / rho
    - U_0_x_expr * U_1_x_expr * h**2 / 6
    - U_0_x_expr * h_x * q_U1 / 2
    - U_1_x_expr**2 * h**2 / 15
    + U_1_x_expr * b_x * q_U0 / 3
    - U_1_x_expr * h_x * q_U1 / 2
    + b_x * h_x * q_U0 * q_U1 / h**2
    - 9 * h_x**2 * q_U1**2 / (10 * h**2)
)
S[6, 0] = 2 * (b_x * q_U0 - q_W0) / h
S[7, 0] = 6 * (b_x * q_U1 - q_W1) / h

## Build SystemModel + BCs (factory)

In [ ]:
def build_8state_sm(bc_list):
    """Build the 8-state SystemModel with the boundary-condition list
    appropriate to the experiment.  All operator tensors are taken
    from the module-level cell above."""
    sm = SystemModel(
        time=t, space=[x], state=list(state), aux_state=list(aux_state),
        parameters=parameters, parameter_values=parameter_values,
        flux=F, hydrostatic_pressure=P, nonconservative_matrix=B,
        source=S, mass_matrix=M,
    )
    sm.equation_names = [
        "mass", "xmom_j0", "xmom_j1", "zmom_j0", "zmom_j1",
        "b_eq", "cont_j1", "cont_j2",
    ]

    position_zs = Zstruct(x=x); position_zs._symbolic_name = "X"
    dX = sp.Symbol("dX", positive=True)
    variables_zs = Zstruct(h=h, q_U0=q_U0, q_U1=q_U1,
                           q_W0=q_W0, q_W1=q_W1, b=b, P_0=P_0, P_1=P_1)
    variables_zs._symbolic_name = "Q"
    aux_variables_zs = Zstruct(b_x=b_x, h_x=h_x, q_U0_x=q_U0_x, q_U1_x=q_U1_x)
    aux_variables_zs._symbolic_name = "Qaux"
    normal_zs = Zstruct(n0=sp.Symbol("n0", real=True))
    normal_zs._symbolic_name = "n"

    sm.boundary_conditions = bc_list.get_boundary_condition_function(
        t, position_zs, dX, variables_zs, aux_variables_zs,
        parameters, normal_zs, function_name="boundary_conditions")
    sm.boundary_gradients = bc_list.get_boundary_gradient_function(
        t, position_zs, dX, variables_zs, aux_variables_zs,
        parameters, normal_zs, function_name="boundary_gradients")
    aux_bc_list = BoundaryConditions([
        AuxBC.Extrapolation(tag="left"),
        AuxBC.Extrapolation(tag="right"),
    ])
    sm.aux_boundary_conditions = aux_bc_list.get_boundary_condition_function(
        t, position_zs, dX, variables_zs, aux_variables_zs,
        parameters, normal_zs, function_name="aux_boundary_conditions")
    sm.initial_conditions = UserFunction(function=lambda x: np.zeros(n_state))
    sm.aux_initial_conditions = UserFunction(
        function=lambda x: np.zeros(len(aux_state)))

    sm.aux_registry = [
        {"kind": "derivative", "name": "b_x", "row": 0,
         "atom": sp.Derivative(b, x, evaluate=False), "aux_symbol": b_x,
         "target_name": "b", "multi_index": (1,),
         "target_kind": "state", "state_index": 5},
        {"kind": "derivative", "name": "h_x", "row": 1,
         "atom": sp.Derivative(h, x, evaluate=False), "aux_symbol": h_x,
         "target_name": "h", "multi_index": (1,),
         "target_kind": "state", "state_index": 0},
        {"kind": "derivative", "name": "q_U0_x", "row": 2,
         "atom": sp.Derivative(q_U0, x, evaluate=False),
         "aux_symbol": q_U0_x,
         "target_name": "q_U0", "multi_index": (1,),
         "target_kind": "state", "state_index": 1},
        {"kind": "derivative", "name": "q_U1_x", "row": 3,
         "atom": sp.Derivative(q_U1, x, evaluate=False),
         "aux_symbol": q_U1_x,
         "target_name": "q_U1", "multi_index": (1,),
         "target_kind": "state", "state_index": 2},
    ]
    sm.assert_diagonal_mass_matrix()
    sm.eigenvalues = None
    return sm

## Test 1 — Lake-at-rest on a Gaussian bump

Initial state `h + b = 0.34` (flat free surface), zero velocity,
zero pressure.  Audusse HR + b-in-NCP should preserve this to
machine precision.

In [ ]:
def test_lake_at_rest(SolverCls=ChorinSplitVAMSolver, n_steps=15):
    bc = BoundaryConditions([Extrapolation(tag="left"),
                             Extrapolation(tag="right")])
    sm = build_8state_sm(bc)
    split = split_simple(sm, [P_0, P_1], sp.Symbol("dt", positive=True))
    mesh = BaseMesh.create_1d(domain=(-1.5, 1.5), n_inner_cells=60)
    solver = SolverCls(split.SM_pred, split.SM_press, split.SM_corr,
        reconstruction_order=1, pressure_tol=1e-9, pressure_maxit=200)
    Q0 = solver.setup_simulation(mesh)
    xc = solver._sim_mesh.cell_centers[0, :solver.nc]
    b_vals = 0.20 * np.exp(-(xc**2) / (2 * 0.20**2))
    Q0[:] = 0
    Q0[0, :] = 0.34 - b_vals
    Q0[5, :] = b_vals
    solver._sim_Q = Q0.copy()
    solver.update_aux_variables()
    dx = float(solver._sim_mesh.cell_volumes[0])
    dt = 0.3 * dx / np.sqrt(9.81 * 0.34)
    print(f"--- LAR ({SolverCls.__name__}), dt = {dt:.5f}")
    print(f"{'step':>4} {'|h-h0|':>10} {'|q_U0|':>10} "
          f"{'|q_U1|':>10} {'|b-b0|':>10}")
    for k in range(n_steps):
        solver.step(dt)
        Q = solver._sim_Q
        print(f"{k+1:>4} {np.max(np.abs(Q[0]-Q0[0])):>10.3e} "
              f"{np.max(np.abs(Q[1])):>10.3e} "
              f"{np.max(np.abs(Q[2])):>10.3e} "
              f"{np.max(np.abs(Q[5]-b_vals)):>10.3e}")


test_lake_at_rest()

## Test 2 — Dam-break over a bump (T = 20)

Initial state: `h = 0.34 − b` for `x < 1.0`, dry `h = 0.015`
downstream.  Left BC: inflow `q_U0 = 0.11197`.  Right BC:
extrapolation.  The simulation reaches a quasi-steady state by
`t ≈ 10`; we compare `η = h + b` at `t = 20` against the digitized
experimental data.

In [ ]:
def test_dam_break(SolverCls=ChorinSplitVAMSolver, T_end=20.0):
    bc = BoundaryConditions([
        Lambda(tag="left", prescribe_fields={
            1: lambda *a: 0.11197, 2: lambda *a: 0.0,
            3: lambda *a: 0.0, 4: lambda *a: 0.0,
            5: lambda *a: 0.20 * np.exp(-(-1.5)**2 / (2 * 0.20**2)),
        }),
        Extrapolation(tag="right"),
    ])
    sm = build_8state_sm(bc)
    split = split_simple(sm, [P_0, P_1], sp.Symbol("dt", positive=True))
    mesh = BaseMesh.create_1d(domain=(-1.5, 1.5), n_inner_cells=60)
    solver = SolverCls(split.SM_pred, split.SM_press, split.SM_corr,
        reconstruction_order=1, pressure_tol=1e-9, pressure_maxit=200)
    Q0 = solver.setup_simulation(mesh)
    xc = solver._sim_mesh.cell_centers[0, :solver.nc]
    b_vals = 0.20 * np.exp(-(xc**2) / (2 * 0.20**2))
    Q0[:] = 0
    Q0[0, :] = np.maximum(np.where(xc < 1.0, 0.34 - b_vals, 0.015), 0.015)
    Q0[5, :] = b_vals
    solver._sim_Q = Q0.copy()
    solver.update_aux_variables()
    dx = float(solver._sim_mesh.cell_volumes[0])
    dt = 0.3 * dx / (np.sqrt(9.81 * 0.34) + 1.0)
    n_steps = int(np.ceil(T_end / dt))
    print(f"--- DAM-BREAK ({SolverCls.__name__}), "
          f"dt = {dt:.5f}, n_steps = {n_steps}")
    print(f"{'step':>5} {'t':>6} {'hmin':>7} {'hmax':>7} {'|q_U0|':>10} "
          f"{'|q_U1|':>10}")
    log_steps = sorted({1, 5, 20, 100, 500, 1000, 2000, n_steps})
    for k in range(n_steps):
        solver.step(dt)
        Q = solver._sim_Q
        if (k + 1) in log_steps:
            print(f"{k+1:>5} {solver._sim_time:>6.3f} {Q[0].min():>7.4f} "
                  f"{Q[0].max():>7.4f} {np.max(np.abs(Q[1])):>10.3e} "
                  f"{np.max(np.abs(Q[2])):>10.3e}")
        if not np.all(np.isfinite(Q)):
            print(f"  BLOWUP at step {k+1}")
            return None
    return solver, xc, b_vals


result = test_dam_break()

## Plot — simulation vs experimental free surface

In [ ]:
if result is not None:
    solver, xc, b_vals = result
    Q = solver._sim_Q
    eta = Q[0] + Q[5]
    fig, ax = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
    ax[0].plot(xc, eta, "b-", lw=1.5, label=f"sim t = {solver._sim_time:.2f} s")
    ax[0].plot(ETA_EXP_X, ETA_EXP_Y, "ko", ms=4, label="experiment")
    ax[0].plot(xc, b_vals, "k-", lw=1.0, alpha=0.5, label="bathymetry")
    ax[0].set_ylabel("free surface η  [m]")
    ax[0].set_title("dam-break over bump, 8-state b-as-state SystemModel")
    ax[0].legend(); ax[0].grid(True, alpha=0.3)
    ax[1].plot(xc, Q[1], "r-", lw=1.5, label="q_U0")
    ax[1].plot(xc, Q[2], "g-", lw=1.5, label="q_U1")
    ax[1].set_xlabel("x  [m]")
    ax[1].set_ylabel("momentum modes  [m²/s]")
    ax[1].legend(); ax[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("dam_break_8state.png", dpi=100, bbox_inches="tight")
    print("saved dam_break_8state.png")

## What this validates

* The OLD-prototype's structural choice (`b` as state, gravity in
  NCP) gives **bit-exact LAR** on a Gaussian bump and **stable
  dam-break propagation** to `T = 20` — both impossible on the
  7-state form even with Audusse HR alone.
* The `split_simple` splitter (introduced in this round) consumes
  any SystemModel whose evolution rows are named by the chain
  convention (`mass`, `xmom_j*`, `zmom_j*`, `b_eq`) and produces a
  three-stage Chorin split.  The fix to
  `build_pressure_elliptic_block` (the conservative-form corrector
  formula) is what brings the q_U / q_W updates onto the correct
  conservative scaling and so removes the spurious `1/h` factor
  that previously masked the bug at `h ≈ 1` but blew up at
  `h ∈ [0.015, 0.34]`.
* The `_build_numerics` overrides on both Chorin solvers now
  exclude `b` from `scaled_q_indices` when bathymetry is a state,
  so Audusse HR no longer rescales it by `h*/h` — that scaling
  was destroying the LAR equilibrium on this 8-state setup.

**Quantitative metric.**  Free-surface elevation at `x = 0` after
the quasi-steady state is reached: simulation ≈ 0.288 m vs.
experiment ≈ 0.305 m at the closest digitized point.  The shape of
the recirculation region behind the bump matches qualitatively;
the residual offset is likely the next refinement target
(mesh resolution? friction? higher-order time integration?).